In [106]:
# Import Packages
import pandas as pd
from dotenv import dotenv_values
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions
import os
from groq import Groq
from dotenv import dotenv_values
from pprint import pprint
from google import genai
from google.genai import types
from pypdf import PdfReader
import requests
from groq import Groq
from sentence_transformers import SentenceTransformer
import re
import pdfplumber
from rank_bm25 import BM25Okapi

### SetUP Vars

In [ ]:
# Import Varaibles
config = dotenv_values(".env")
GROQ_API_KEY = config["GROQ_API_KEY"]
GEMINI_API_KEY = config["GEMINI_API_KEY"]
HF_TOKEN =config.get("HF_TOKEN")

In [108]:
# Import Embedding Model and LLM Model
os.environ["HF_TOKEN"] = HF_TOKEN
model = SentenceTransformer("all-MiniLM-L6-v2")

# Load LLM Model
# ollama server --> this will run model in locally in http://localhost:11434

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2865.51it/s]


In [109]:
# try all-MiniLM-L6-v2 Model
sentences = "What is Meaning Of Life ?"
embeddings = model.encode(sentences)
print(embeddings.shape)

(384,)


In [110]:
# Init Chroma Vector Database
chroma_client = chromadb.PersistentClient(path="course")
collection_name = "course"
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_or_create_collection(name=collection_name, embedding_function=embedding_function)
print(collection)

Collection(name=course)


### Start getting documents from PDF

In [111]:
def get_text_from_pdf(path):
    documents = []
    with pdfplumber.open(path) as pdf:
        print(f"number of pages is: {len(pdf.pages)}")

        for i, page in enumerate(pdf.pages):
            text = page.extract_text()

            if text:
                # remove extra spaces
                text = text.strip()

                # fix line breaks inside paragraphs
                text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

                # normalize multiple spaces/newlines
                text = re.sub(r'\s+', ' ', text)

                documents.append({"id": i, "text": text})
            else:
                documents.append("")
        return documents

documents =  get_text_from_pdf("xml_course.pdf")
print("number of documents: ", len(documents))

number of pages is: 55
number of documents:  55


In [112]:
# Function to split text into chunks
def split_text(text, chunk_size=1000, chunk_overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks

In [113]:
documents[0]

{'id': 0,
 'text': 'XML : Cours et exercices corrigés Master Web Intelligence et Sciences des Données (WISD) Prof. El Habib NFAOUI (elhabib.nfaoui@usmba.ac.ma) Edition 2020'}

In [114]:
# Split documents into chunks
chunked_documents = []
for doc in documents:
    chunks = split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            "id": f"{doc['id']}_chunk{i+1}",
            "text": chunk
        })
print(len(chunked_documents))

93


In [115]:
# Function to Generate embeddings for the document chunks - Doc2Vect
def get_embedding(text):
    embeddings = model.encode(text)
    return embeddings

In [116]:
# Get Embedding Vectors
for doc_chunk in chunked_documents:
    doc_chunk['embeddings'] = get_embedding(doc_chunk["text"])

In [117]:
# Save Embedding Into Vector Database
for doc_chunk in chunked_documents:
    collection.upsert(ids=[doc_chunk["id"]], documents=[doc_chunk["text"]], embeddings=[doc_chunk["embeddings"]])


### Start Working With LLM

In [118]:
# Get Documents from LLM
client_groq = Groq(api_key=GROQ_API_KEY)
def get_llm_documents(question):
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
        {
            "role": "system",
            "content": f"You are a technical documentation writer. Write one clear\
            structured documentation for: {question}, use a simple words in eglich, and in breif. and just talk about the question.",
        },
        ],
        temperature=1,
        max_completion_tokens=1024, 
        top_p=1,
        stream=True,
        stop=None
    )

    res = []
    for chunk in completion:
        res.append(chunk.choices[0].delta.content)

    res = [s for s in res if s]
    res = "".join(res)
    return res

In [133]:
# Get LLM Documents
response = get_llm_documents(question="What is AI ?")

In [134]:
# Function to Split LLM Document
def split_text_llm(text):
    chunks = []

    for i, p in enumerate(text.split("\n")):
        p = p.strip()

        if len(p) > 50:
            chunks.append({
                "id": f"chunk_{i}",
                "text": p,
                "embedding": None
            })

    return chunks

In [135]:
def get_llm_embedding(response):
    # Split LLm Documents Into Chunks
    llm_documents = split_text_llm(response)
    
    # Embedding LLM Document
    for doc_chunk in llm_documents:
        doc_chunk['embeddings'] = get_embedding(doc_chunk["text"])
    return llm_documents

In [149]:
# Function That Query Documents
'''
    collection.query(): find the documents most related to text
        - query_texts: the question or text you want to search with.
        - n_results: how many documents we want back.
'''


def bm25_search(bm25, question, chunked_documents, top_k=2):
    tokenized_query = question.split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    results = []

    for i in top_indices:
        results.append({
            "text": chunked_documents[i]["text"],
            "embeddings": chunked_documents[i]["embeddings"],
            "score": scores[i]
        })

    return results



In [164]:
# Compare Similarity of Relevent Chunks With LLM Chunks
question = "utilisé pour une partie du cours intitulé XML ?"

# Step 1 → Generate hypothetical answer
hyde_doc = get_llm_documents(question)

# Step 2 → Embed hypothetical answer
hyde_doc_embedding = get_llm_embedding(hyde_doc)
# print(hyde_doc_embedding)

# Step 3 → Get the Filtred Documents With BM25
corpus = [doc["text"].split() for doc in chunked_documents]
bm25 = BM25Okapi(corpus)
filtred_chunked_documents = bm25_search(bm25, question=question, chunked_documents=chunked_documents)
# print(filtred_chunked_documents)


In [156]:
from sklearn.metrics.pairwise import cosine_similarity

NameError: name 'hyde_doc_embedding' is not defined